In [10]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [11]:
!ls -lh /content/drive/MyDrive/bridge_project

ls: /content/drive/MyDrive/bridge_project
lrw------- 1 root root 0 Nov  6 13:11 /content/drive/MyDrive/bridge_project -> /content/drive/.shortcut-targets-by-id/1aGl3EY-WpuFNu8b4E2TOf7qAkYQW9ctm/bridge_project


In [13]:
!ls -lh /content/drive/MyDrive/bridge_project/

total 913M
-r-------- 1 root root 4.6M Nov  1 13:23 all_polygons.csv
-r-------- 1 root root  94M Nov  3 06:58 baseline_unet_r34_FINAL.pth
-r-------- 1 root root  103 Nov  3 06:56 best_thresholds.json
-r-------- 1 root root  94M Nov  2 10:36 best_unet_r34.pth
dr-x------ 2 root root 4.0K Nov  1 13:23 figures
-r-------- 1 root root 3.0K Nov  2 10:36 history.csv
-r-------- 1 root root 722M Nov  1 13:46 processed.zip


In [14]:
!ls /content/processed || unzip -q /content/drive/MyDrive/bridge_project/processed.zip -d /content/

!ls /content/processed

!pip install -q segmentation-models-pytorch albumentations opencv-python-headless scikit-image

ls: cannot access '/content/processed': No such file or directory
test  train  val


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
PROC = Path("/content/processed")
BASE = Path("/content/drive/MyDrive/bridge_project")            # shared by the owner: READ ONLY
EXP_ROOT = Path("/content/drive/MyDrive/bridge_experiments")    # YOUR OWN results folder
EXP_ROOT.mkdir(parents=True, exist_ok=True)
TARGET = ["Crack", "Spalling", "Rust", "Efflorescence", "Cavity", "ExposedRebars"]
NC = len(TARGET); SIZE = 512
DEV = "cuda" if torch.cuda.is_available() else "cpu"
COLORS = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),(0,255,255)]

def decode(m): return np.stack([(m >> c) & 1 for c in range(NC)], -1).astype(np.float32)
def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything()

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_train = A.Compose([A.HorizontalFlip(p=0.5), A.Affine(scale=(0.85, 1.15), rotate=(-10, 10),
p=0.5),
                      A.RandomBrightnessContrast(p=0.5), A.HueSaturationValue(p=0.3),
                      A.Normalize(MEAN, STD), ToTensorV2(transpose_mask=True)])
tf_val = A.Compose([A.Normalize(MEAN, STD), ToTensorV2(transpose_mask=True)])

class DaclDataset(Dataset):
    def __init__(self, split, tf):
        self.split, self.tf = split, tf
        self.ids = sorted(p.stem for p in (PROC/split/"img").glob("*.jpg"))
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        s = self.ids[i]
        img = cv2.cvtColor(cv2.imread(str(PROC/self.split/"img"/f"{s}.jpg")),
cv2.COLOR_BGR2RGB)
        m = cv2.imread(str(PROC/self.split/"mask"/f"{s}.png"), cv2.IMREAD_GRAYSCALE)
        out = self.tf(image=img, mask=decode(m))
        return out["image"], out["mask"]

PIN = (DEV == "cuda")
train_dl = DataLoader(DaclDataset("train", tf_train), batch_size=8, shuffle=True,
num_workers=2, pin_memory=PIN, drop_last=True)
val_dl   = DataLoader(DaclDataset("val",   tf_val),   batch_size=8, shuffle=False,
num_workers=2, pin_memory=PIN)
test_dl  = DataLoader(DaclDataset("test",  tf_val),   batch_size=8, shuffle=False,
num_workers=2, pin_memory=PIN)

bce = nn.BCEWithLogitsLoss()
dice = smp.losses.DiceLoss(mode="multilabel", from_logits=True)
def loss_fn(logits, y): return bce(logits, y) + dice(logits, y)

@torch.no_grad()
def evaluate(model, loader, thr=0.5):
    model.eval()
    thr = torch.as_tensor(thr, device=DEV, dtype=torch.float32).view(1, -1, 1, 1) if not np.isscalar(thr) else thr
    tp = torch.zeros(NC, device=DEV); fp = torch.zeros(NC, device=DEV); fn = torch.zeros(NC,
device=DEV)
    for x, y in loader:
        x, y = x.to(DEV), y.to(DEV)
        with torch.autocast("cuda", enabled=(DEV == "cuda")):
            p = torch.sigmoid(model(x))
        pr = (p > thr).float()
        tp += (pr * y).sum((0, 2, 3)); fp += (pr * (1 - y)).sum((0, 2, 3)); fn += ((1 - pr) * y).sum((0, 2,
3))
    iou = tp / (tp + fp + fn + 1e-7); dsc = 2 * tp / (2 * tp + fp + fn + 1e-7)
    prec = tp / (tp + fp + 1e-7); rec = tp / (tp + fn + 1e-7)
    return {k: v.cpu().numpy() for k, v in dict(iou=iou, dice=dsc, precision=prec,
recall=rec).items()}

th = json.load(open(BASE/"best_thresholds.json"))
best_thr_baseline = np.array([th[t] for t in TARGET])
print({s: len(list((PROC/s/"img").glob("*.jpg"))) for s in ["train", "val", "test"]}, "| device:", DEV)
print("thresholds:", dict(zip(TARGET, best_thr_baseline)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'train': 5245, 'val': 582, 'test': 975} | device: cuda
thresholds: {'Crack': np.float64(0.7), 'Spalling': np.float64(0.2), 'Rust': np.float64(0.1), 'Efflorescence': np.float64(0.3), 'Cavity': np.float64(0.6), 'ExposedRebars': np.float64(0.1)}


In [2]:
base_model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3,
classes=NC, activation=None).to(DEV)
base_model.load_state_dict(torch.load(BASE/"best_unet_r34.pth", map_location=DEV));
base_model.eval()
r = evaluate(base_model, val_dl, thr=best_thr_baseline)
print("baseline val mIoU:", round(float(r["iou"].mean()), 3), "(you should see about 0.326)")
del base_model
if DEV == "cuda": torch.cuda.empty_cache()

baseline val mIoU: 0.326 (you should see about 0.326)


In [3]:
OUT = EXP_ROOT/"baseline_eval"; OUT.mkdir(parents=True, exist_ok=True)

model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3,
classes=NC, activation=None).to(DEV)
model.load_state_dict(torch.load(BASE/"best_unet_r34.pth", map_location=DEV)); model.eval()

t0 = time.time()
test_t = evaluate(model, test_dl, thr=best_thr_baseline)
test_5 = evaluate(model, test_dl, thr=0.5) if DEV == "cuda" else None

cols = {"IoU (tuned thr)": test_t["iou"], "Dice": test_t["dice"],
        "Precision": test_t["precision"], "Recall": test_t["recall"]}
if test_5 is not None: cols["IoU (thr 0.5)"] = test_5["iou"]
tbl = pd.DataFrame(cols, index=TARGET).round(3)
print(tbl)
print("Test mIoU (tuned thresholds):", round(float(test_t["iou"].mean()), 3), f"| {(time.time()-t0)/60:.1f} min")
tbl.to_csv(OUT/"test_results.csv")

summary = dict(experiment="unet_r34 (baseline)",
               val_miou_tuned=0.326,        # measured by the owner in Step 17
               test_miou_tuned=float(test_t["iou"].mean()),
               test_miou_thr050=(float(test_5["iou"].mean()) if test_5 is not None else None),
               test_iou_per_class=dict(zip(TARGET, [round(float(v), 3) for v in test_t["iou"]])),
               thresholds=dict(zip(TARGET, [round(float(v), 2) for v in best_thr_baseline])),
               minutes_per_epoch=3.5,
               params_millions=round(sum(p.numel() for p in model.parameters())/1e6, 1))
json.dump(summary, open(OUT/"results.json", "w"), indent=2)

               IoU (tuned thr)   Dice  Precision  Recall  IoU (thr 0.5)
Crack                    0.241  0.388      0.465   0.333          0.242
Spalling                 0.359  0.528      0.579   0.486          0.355
Rust                     0.396  0.567      0.575   0.559          0.397
Efflorescence            0.287  0.446      0.537   0.382          0.284
Cavity                   0.139  0.245      0.217   0.280          0.140
ExposedRebars            0.377  0.548      0.540   0.557          0.375
Test mIoU (tuned thresholds): 0.3 | 0.7 min


In [4]:
!ls /content/drive/MyDrive/fpn_r34
!ls /content/drive/MyDrive/deeplabv3p_r34

best.pth  history.csv  last.pth  results.json
best.pth  history.csv  last.pth  results.json


In [5]:
rows = [json.load(open(EXP_ROOT/"baseline_eval"/"results.json"))]
for name in ["fpn_r34", "deeplabv3p_r34"]:
    p = Path(f"/content/drive/MyDrive/{name}/results.json")
    if p.exists(): rows.append(json.load(open(p)))
    else: print("not ready yet:", name)

cmp = pd.DataFrame([{"model": r["experiment"], "val mIoU (tuned)": round(r["val_miou_tuned"],
3), "test mIoU (tuned)": round(r["test_miou_tuned"], 3), "min/epoch": r["minutes_per_epoch"],"params (M)": r["params_millions"], **r["test_iou_per_class"]} for r in rows])
print(cmp.to_string(index=False))
winner = cmp.sort_values("val mIoU (tuned)", ascending=False).iloc[0]["model"]
print("\nWinner by VALIDATION mIoU:", winner)
cmp.to_csv(EXP_ROOT/"baseline_eval"/"model_comparison.csv", index=False)

              model  val mIoU (tuned)  test mIoU (tuned)  min/epoch  params (M)  Crack  Spalling  Rust  Efflorescence  Cavity  ExposedRebars
unet_r34 (baseline)             0.326              0.300       3.50        24.4  0.241     0.359 0.396          0.287   0.139          0.377
            fpn_r34             0.339              0.306       3.16        23.2  0.249     0.374 0.370          0.312   0.147          0.381
     deeplabv3p_r34             0.333              0.301       3.05        22.4  0.241     0.375 0.389          0.285   0.177          0.338

Winner by VALIDATION mIoU: fpn_r34


In [6]:
def build(name):
    kw = dict(in_channels=3, classes=NC, activation=None)
    if name.startswith("unet_r34"):  return smp.Unet(encoder_name="resnet34", encoder_weights=None, **kw)
    if name == "fpn_r34":            return smp.FPN(encoder_name="resnet34", encoder_weights=None, **kw)
    if name == "deeplabv3p_r34":     return smp.DeepLabV3Plus(encoder_name="resnet34", encoder_weights=None, **kw)

CKPT_OF = {"unet_r34 (baseline)": BASE/"best_unet_r34.pth",
           "fpn_r34": Path("/content/drive/MyDrive/fpn_r34/best.pth"),
           "deeplabv3p_r34": Path("/content/drive/MyDrive/deeplabv3p_r34/best.pth")}

thr_dict = [r for r in rows if r["experiment"] == winner][0]["thresholds"]
best_thr = np.array([thr_dict[t] for t in TARGET])
model = build(winner).to(DEV)
model.load_state_dict(torch.load(CKPT_OF[winner], map_location=DEV)); model.eval()

r = evaluate(model, val_dl, thr=best_thr)
print("Loaded:", winner, "| val mIoU:", round(float(r["iou"].mean()), 3), "(should match the table, within about 0.002)")

Loaded: fpn_r34 | val mIoU: 0.339 (should match the table, within about 0.002)


In [7]:
FIG = EXP_ROOT/"figures"; FIG.mkdir(parents=True, exist_ok=True)

def show_pred(split, k, save=None):
    ds = DaclDataset(split, tf_val)
    x, y = ds[k]
    with torch.no_grad():
        p = torch.sigmoid(model(x[None].to(DEV)))[0].cpu().numpy()
    pr = (p > np.array(best_thr)[:, None, None])
    im = cv2.cvtColor(cv2.imread(str(PROC/split/"img"/f"{ds.ids[k]}.jpg")), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, t, m in zip(ax, ["Image", "Ground truth", "Prediction"], [None, y.numpy(), pr]):
        o = im.copy()
        if m is not None:
            for c in range(NC): o[m[c] > 0] = COLORS[c]
            o = cv2.addWeighted(im, .5, o, .5, 0)
        a.imshow(o); a.set_title(t); a.axis("off")
    if save: plt.savefig(save, dpi=120, bbox_inches="tight")
    plt.show()

random.seed(1)
for k in random.sample(range(len(test_dl.dataset)), 8):
    show_pred("test", k, save=FIG/f"pred_{winner.split()[0]}_{k}.png")

Output hidden; open in https://colab.research.google.com to view.